# M2 학습예산 3시드 원인진단 집계 — seeds 42·43·44

학습은 하지 않습니다. 사전에 고정한 100·300 epoch에서 동일 seed의 `M2−M1`을 비교해 다음을 구분합니다.

1. seed 42만 역전했는가
2. 장기학습의 Top-10 회복이 3시드에서 반복되는가
3. @20·@50 및 고CLV 경제지표 하락도 반복되는가

중간 최고 checkpoint는 선택하지 않으며 유의성·최종 성공·CLV 귀속을 주장하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import pandas as pd

ROOT=Path('/content/drive/MyDrive/논문/data')
FOLDERS={
 42: ROOT/'results_v3_dunnhumby_clv_m2_capacity_search_v1',
 43: ROOT/'results_v3_dunnhumby_clv_m2_training_budget_seed43_v1',
 44: ROOT/'results_v3_dunnhumby_clv_m2_training_budget_seed44_v1',
}
frames=[]
for seed,folder in FOLDERS.items():
    files=list(folder.glob('clv_m2_capacity_search_*_curve.csv'))
    assert len(files)==1, f'seed {seed} curve CSV 개수={len(files)}: {files}'
    frame=pd.read_csv(files[0])
    assert set(frame.seed.astype(int))=={seed}
    assert set(frame.id_dim.astype(int))=={64}
    assert set(frame.pref_reg.astype(float))=={0.001}
    frame.insert(0,'audit_seed',seed); frames.append(frame)
curve=pd.concat(frames,ignore_index=True)
print('rows, columns:',curve.shape)

In [ ]:
METRICS=[
 'recall@10','ndcg@10','recall@20','ndcg@20','recall@50','ndcg@50',
 'price_purchase_amount_weighted_hit@10','vndcg@10',
 'price_purchase_amount_weighted_hit@20','vndcg@20',
 'price_purchase_amount_weighted_hit@50','vndcg@50',
 '고CLV_recall@10','고CLV_ndcg@10','고CLV_revenue@10','고CLV_vndcg@10',
 '고CLV_recall@20','고CLV_ndcg@20','고CLV_revenue@20','고CLV_vndcg@20',
 '고CLV_recall@50','고CLV_ndcg@50','고CLV_revenue@50','고CLV_vndcg@50',
]
assert not [m for m in METRICS if m not in curve], '필수지표 누락'
rows=[]
for (seed,epoch),part in curve[curve.epoch.isin([100,300])].groupby(['audit_seed','epoch']):
    m1=part[part.model_id.eq('m1_bpr_k1')].iloc[0]
    m2=part[part.model_id.eq('m2_nv_history_fit_bpr_k1')].iloc[0]
    for metric in METRICS:
        base,value=float(m1[metric]),float(m2[metric])
        rows.append({'seed':int(seed),'epoch':int(epoch),'metric':metric,'M1':base,'M2':value,
                     'delta':value-base,'relative_change_pct':100*(value-base)/base if base else float('nan')})
comparison=pd.DataFrame(rows)
print(comparison.to_string(index=False))

In [ ]:
# 100→300에서 M2-M1 격차가 회복됐는지 seed별로 본다.
pivot=comparison.pivot_table(index=['seed','metric'],columns='epoch',values='delta').reset_index()
pivot['gap_change_300_minus_100']=pivot[300]-pivot[100]
pivot['m2_beats_m1_at_300']=pivot[300]>0
summary=(pivot.groupby('metric')
         .agg(seeds_gap_improved=('gap_change_300_minus_100','sum'),
              seeds_m2_beats_m1_at_300=('m2_beats_m1_at_300','sum'),
              mean_gap_change=('gap_change_300_minus_100','mean'),
              mean_gap_at_300=(300,'mean')).reset_index())
print('3시드 판독표')
print(summary.to_string(index=False))
out=ROOT/'m2_training_budget_seed42_43_44_diagnostic.csv'
comparison.to_csv(out,index=False)
print('저장:',out)